In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

In [ ]:
transactions  = pd.read_csv('/content/Transactions.csv')
products      = pd.read_csv('/content/Products.csv')
log_inventory = pd.read_csv('/content/Logs.csv')
suppliers     = pd.read_csv('/content/Suppliers.csv')
sellers       = pd.read_csv('/content/Sellers.csv')

print("Transactions :", transactions.shape)
print("Products     :", products.shape)
print("LogInventory :", log_inventory.shape)
print("Suppliers    :", suppliers.shape)
print("Sellers      :", sellers.shape)

Transactions : (30000, 14)
Products     : (40, 11)
LogInventory : (10000, 9)
Suppliers    : (8, 4)
Sellers      : (6, 5)


In [ ]:
print("tabel Transactions")
print(transactions.columns.tolist())
print("Tabel Products")
print(products.columns.tolist())
print("Tabel LogInventory")
print(log_inventory.columns.tolist())

tabel Transactions
['id_transaksi', 'id_seller', 'id_produk', 'tanggal', 'jam_transaksi', 'jenis_transaksi', 'kategori', 'qty', 'harga_satuan', 'total_harga', 'metode_pembayaran', 'event', 'diskon', 'status_transaksi']
Tabel Products
['id_produk', 'id_seller', 'id_supplier', 'nama_produk', 'kategori_produk', 'harga_jual', 'harga_modal', 'stok_awal', 'minimum_stok', 'status_produk', 'tanggal_input_produk']
Tabel LogInventory
['id_log', 'id_produk', 'id_seller', 'tanggal', 'jenis_perubahan', 'jumlah', 'stok_sebelum', 'stok_sesudah', 'alasan']


In [ ]:
#Mmrge Transactions + Products + Sellers
data = (
    transactions
    .merge(products,   on=['id_produk', 'id_seller'], how='left')
    .merge(suppliers,  on='id_supplier',              how='left')
    .merge(sellers,    on='id_seller',                how='left')
)

print("Shape setelah merge:", data.shape)
print(data.isnull().sum())

Shape setelah merge: (30000, 30)
id_transaksi            0
id_seller               0
id_produk               0
tanggal                 0
jam_transaksi           0
jenis_transaksi         0
kategori                0
qty                     0
harga_satuan            0
total_harga             0
metode_pembayaran       0
event                   0
diskon                  0
status_transaksi        0
id_supplier             0
nama_produk             0
kategori_produk         0
harga_jual              0
harga_modal             0
stok_awal               0
minimum_stok            0
status_produk           0
tanggal_input_produk    0
nama_supplier           0
kategori_supplier       0
lokasi                  0
nama_usaha              0
jenis_usaha             0
lokasi_usaha            0
tanggal_bergabung       0
dtype: int64


In [ ]:
JENIS_JUAL   = ['pemasukan']
STATUS_VALID = ['berhasil']

print("Nilai unik jenis_transaksi:", data['jenis_transaksi'].unique())
print("Nilai unik status_transaksi:", data['status_transaksi'].unique())

Nilai unik jenis_transaksi: ['Pemasukan' 'Pengeluaran']
Nilai unik status_transaksi: ['Berhasil' 'Dibatalkan']


In [ ]:
data = data[
    data['jenis_transaksi'].str.lower().isin(['pemasukan']) &
    data['status_transaksi'].str.lower().isin(['berhasil'])
].copy()

data['tanggal'] = pd.to_datetime(data['tanggal'])
data = data.dropna(subset=['id_seller', 'kategori_produk', 'qty', 'tanggal'])
data['qty'] = pd.to_numeric(data['qty'], errors='coerce').fillna(0)

print("Shape setelah filter:", data.shape)

Shape setelah filter: (26974, 30)


In [ ]:
log_inventory['tanggal'] = pd.to_datetime(log_inventory['tanggal'])

arah = {
    'Restock'          :  1,
    'Retur Supplier'   :  1,
    'Penjualan'        : -1,
    'Barang Rusak'     : -1,
    'Kadaluarsa'       : -1,
    'Penyesuaian'      :  0,
}

log_inventory['arah'] = log_inventory['jenis_perubahan'].map(arah).fillna(0)
log_inventory['perubahan_net'] = log_inventory['jumlah'] * log_inventory['arah']

net_per_produk = (
    log_inventory
    .groupby('id_produk')['perubahan_net']
    .sum()
    .reset_index()
    .rename(columns={'perubahan_net': 'total_perubahan'})
)

stock_info = products[['id_produk', 'id_seller', 'kategori_produk', 'minimum_stok', 'stok_awal']].merge(
    net_per_produk, on='id_produk', how='left'
)
stock_info['total_perubahan'] = stock_info['total_perubahan'].fillna(0)
stock_info['stok_aktual']     = (stock_info['stok_awal'] + stock_info['total_perubahan']).clip(lower=0)

print("Contoh data stok aktual per produk:")
print(stock_info[['id_produk', 'stok_awal', 'total_perubahan', 'stok_aktual', 'minimum_stok']].head(10).to_string(index=False))

Contoh data stok aktual per produk:
id_produk  stok_awal  total_perubahan  stok_aktual  minimum_stok
     P001        757            40246        41003           151
     P002        825            38605        39430           165
     P003        752            35904        36656           150
     P004        744            43156        43900           149
     P005        715            33611        34326           143
     P006       3534            22751        26285           707
     P007       2907            26220        29127           581
     P008       3616            20658        24274           723
     P009       3706            18737        22443           741
     P010       2513            19784        22297           503


In [ ]:
#buat stok per seller per kategori
stock_per_seller_cat = (
    stock_info
    .groupby(['id_seller', 'kategori_produk'])
    .agg(
        stok_aktual  = ('stok_aktual',  'sum'),
        minimum_stok = ('minimum_stok', 'sum')
    )
    .reset_index()
)

print("Contoh stok per seller per kategori:")
print(stock_per_seller_cat.head(10).to_string(index=False))

Contoh stok per seller per kategori:
id_seller kategori_produk  stok_aktual  minimum_stok
     U001         Minuman       195315           758
     U002         Sembako       124426          3255
     U002           Snack       100102          1588
     U003     Frozen Food        94934          1063
     U003         Makanan        99330           857
     U004          Bakery       192820           721
     U005         Laundry       213384          3507
     U006         Fashion       199314           513


In [ ]:
#setiap kombinasi seler + kategori punya 1 model sendiri
seller_cat_pairs = (
    data
    .groupby(['id_seller', 'kategori_produk'])
    .size()
    .reset_index(name='total_transaksi')
    .sort_values('total_transaksi', ascending=False)
    .reset_index(drop=True)
)

print(f"Total kombinasi seller × kategori: {len(seller_cat_pairs)}")
print(seller_cat_pairs.head(20).to_string(index=False))

Total kombinasi seller × kategori: 8
id_seller kategori_produk  total_transaksi
     U006         Fashion             4614
     U004          Bakery             4544
     U001         Minuman             4488
     U005         Laundry             4406
     U002           Snack             2318
     U002         Sembako             2258
     U003         Makanan             2180
     U003     Frozen Food             2166


In [ ]:
WINDOW_SIZE   = 8
MIN_WEEKS     = 52
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15

EPOCHS        = 80
BATCH_SIZE    = 8
PATIENCE      = 8

os.makedirs('models',  exist_ok=True)
os.makedirs('scalers', exist_ok=True)

In [ ]:
def create_sequences(scaled_data, window_size):
    X, y = [], []
    for i in range(len(scaled_data) - window_size):
        X.append(scaled_data[i : i + window_size])
        y.append(scaled_data[i + window_size])
    return np.array(X), np.array(y)


def split_sequences(X, y, train_ratio, val_ratio):
    n         = len(X)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return (
        X[:train_end],        y[:train_end],
        X[train_end:val_end], y[train_end:val_end],
        X[val_end:],          y[val_end:]
    )


def build_model(window_size):
    inputs  = Input(shape=(window_size, 1))
    x       = LSTM(64)(inputs)
    x       = Dropout(0.2)(x)
    x       = Dense(32, activation='relu')(x)
    outputs = Dense(1)(x)
    model   = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def safe_model_key(seller_id, category):
    """Buat nama file yang aman dari karakter spesial."""
    safe_cat = category.replace('/', '_').replace(' ', '_').replace('\\', '_')
    return f"{seller_id}__{safe_cat}"


class TrainingLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            val_loss = logs.get('val_loss', float('nan'))
            print(f"  Epoch {epoch+1:3d} | loss: {logs['loss']:.4f} | val_loss: {val_loss:.4f}")

In [ ]:
def classify_restock(predicted_demand, current_stock, minimum_stok):

    if predicted_demand <= 0:
        return 'Stok Aman'

    if current_stock <= minimum_stok:
        return 'Restock Segera'

    coverage_weeks = current_stock / predicted_demand

    if coverage_weeks < 1.5:
        return 'Restock Segera'
    elif coverage_weeks < 3.0:
        return 'Monitor'
    else:
        return 'Stok Aman'

In [ ]:
training_results = []

def train_model(seller_id, category):
    key = safe_model_key(seller_id, category)
    label = f"Seller {seller_id} | {category}"
    print(f"\n{'='*60}")
    print(f"  TRAINING: {label}")
    print(f"{'='*60}")

    mask = (
        (data['id_seller']        == seller_id) &
        (data['kategori_produk']  == category)
    )
    subset = data[mask].copy()

    weekly = (
        subset
        .set_index('tanggal')
        .resample('W')['qty']
        .sum()
        .reset_index()
        .rename(columns={'qty': 'qty_total'})
    )
    print(f"  Data mingguan: {len(weekly)} minggu")

    if len(weekly) < MIN_WEEKS:
        print(f"  [SKIP] Data kurang ({len(weekly)} < {MIN_WEEKS} minggu)")
        return None

    sales = weekly[['qty_total']]

    train_size = int(len(sales) * TRAIN_RATIO)
    scaler = MinMaxScaler()
    scaler.fit(sales[:train_size])
    scaled_all = scaler.transform(sales)

    X, y = create_sequences(scaled_all, WINDOW_SIZE)
    X_train, y_train, X_val, y_val, X_test, y_test = split_sequences(
        X, y, TRAIN_RATIO, VAL_RATIO
    )
    print(f"  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

    if len(X_val) == 0 or len(X_test) == 0:
        print("  [SKIP] Val atau Test kosong setelah split")
        return None

    model = build_model(WINDOW_SIZE)
    early_stop = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=[early_stop, TrainingLogger()],
        verbose=0
    )

    preds_scaled = model.predict(X_test, verbose=0)
    preds_actual = scaler.inverse_transform(preds_scaled)
    y_actual     = scaler.inverse_transform(y_test)

    mae  = mean_absolute_error(y_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, preds_actual))
    print(f"  MAE: {mae:.4f} | RMSE: {rmse:.4f}")

    model.save(f'models/{key}.keras')
    joblib.dump(scaler, f'scalers/{key}_scaler.pkl')
    print(f"  Tersimpan: models/{key}.keras")

    result = {
        'seller_id'  : seller_id,
        'kategori'   : category,
        'model_key'  : key,
        'data_minggu': len(weekly),
        'train_size' : len(X_train),
        'val_size'   : len(X_val),
        'test_size'  : len(X_test),
        'mae'        : round(mae, 4),
        'rmse'       : round(rmse, 4),
    }
    training_results.append(result)
    return result

In [ ]:
training_results = []

for _, row in seller_cat_pairs.iterrows():
    train_model(row['id_seller'], row['kategori_produk'])


  TRAINING: Seller U006 | Fashion
  Data mingguan: 88 minggu
  Train: (56, 8, 1) | Val: (12, 8, 1) | Test: (12, 8, 1)
  Epoch  10 | loss: 0.0446 | val_loss: 0.0110
  MAE: 11.4444 | RMSE: 13.7706
  Tersimpan: models/U006__Fashion.keras

  TRAINING: Seller U004 | Bakery
  Data mingguan: 118 minggu
  Train: (77, 8, 1) | Val: (16, 8, 1) | Test: (17, 8, 1)
  Epoch  10 | loss: 0.0444 | val_loss: 0.0334
  Epoch  20 | loss: 0.0455 | val_loss: 0.0335
  Epoch  30 | loss: 0.0440 | val_loss: 0.0326
  Epoch  40 | loss: 0.0410 | val_loss: 0.0323
  Epoch  50 | loss: 0.0461 | val_loss: 0.0326
  MAE: 23.2763 | RMSE: 29.5936
  Tersimpan: models/U004__Bakery.keras

  TRAINING: Seller U001 | Minuman
  Data mingguan: 156 minggu
  Train: (103, 8, 1) | Val: (22, 8, 1) | Test: (23, 8, 1)
  Epoch  10 | loss: 0.0553 | val_loss: 0.0526
  Epoch  20 | loss: 0.0564 | val_loss: 0.0514
  Epoch  30 | loss: 0.0556 | val_loss: 0.0514
  Epoch  40 | loss: 0.0540 | val_loss: 0.0511
  Epoch  50 | loss: 0.0540 | val_loss: 0

  MAE: 15.3897 | RMSE: 18.7202
  Tersimpan: models/U002__Snack.keras

  TRAINING: Seller U002 | Sembako
  Data mingguan: 149 minggu
  Train: (98, 8, 1) | Val: (21, 8, 1) | Test: (22, 8, 1)
  Epoch  10 | loss: 0.0422 | val_loss: 0.0367
  Epoch  20 | loss: 0.0431 | val_loss: 0.0354
  Epoch  30 | loss: 0.0418 | val_loss: 0.0354
  Epoch  40 | loss: 0.0402 | val_loss: 0.0337
  Epoch  50 | loss: 0.0402 | val_loss: 0.0334
  Epoch  60 | loss: 0.0413 | val_loss: 0.0329
  Epoch  70 | loss: 0.0405 | val_loss: 0.0341


  MAE: 18.7006 | RMSE: 22.4404
  Tersimpan: models/U002__Sembako.keras

  TRAINING: Seller U003 | Makanan
  Data mingguan: 129 minggu
  Train: (84, 8, 1) | Val: (18, 8, 1) | Test: (19, 8, 1)
  MAE: 15.3072 | RMSE: 18.6870
  Tersimpan: models/U003__Makanan.keras

  TRAINING: Seller U003 | Frozen Food
  Data mingguan: 129 minggu
  Train: (84, 8, 1) | Val: (18, 8, 1) | Test: (19, 8, 1)
  MAE: 16.5441 | RMSE: 20.7563
  Tersimpan: models/U003__Frozen_Food.keras


In [ ]:
#hasil training
if training_results:
    summary_df = (
        pd.DataFrame(training_results)
        .sort_values('rmse')
        .reset_index(drop=True)
    )
    print(f"\nModel berhasil dilatih : {len(summary_df)}")
    print(f"Kombinasi dilewati     : {len(seller_cat_pairs) - len(summary_df)}")
    print("\n" + summary_df.to_string(index=False))
else:
    print("Tidak ada model yang berhasil dilatih.")


Model berhasil dilatih : 8
Kombinasi dilewati     : 0

seller_id    kategori         model_key  data_minggu  train_size  val_size  test_size     mae    rmse
     U006     Fashion     U006__Fashion           88          56        12         12 11.4444 13.7706
     U001     Minuman     U001__Minuman          156         103        22         23 14.7869 17.1407
     U003     Makanan     U003__Makanan          129          84        18         19 15.3072 18.6870
     U002       Snack       U002__Snack          149          98        21         22 15.3897 18.7202
     U003 Frozen Food U003__Frozen_Food          129          84        18         19 16.5441 20.7563
     U002     Sembako     U002__Sembako          149          98        21         22 18.7006 22.4404
     U004      Bakery      U004__Bakery          118          77        16         17 23.2763 29.5936
     U005     Laundry     U005__Laundry          101          65        14         14 45.2988 59.5647


In [ ]:
def predict_and_recommend(seller_id, category):

    key         = safe_model_key(seller_id, category)
    model_path  = f'models/{key}.keras'
    scaler_path = f'scalers/{key}_scaler.pkl'

    if not os.path.exists(model_path):
        return None

    model  = load_model(model_path)
    scaler = joblib.load(scaler_path)

    mask = (
        (data['id_seller']       == seller_id) &
        (data['kategori_produk'] == category)
    )
    weekly = (
      data[mask]
      .set_index('tanggal')
      .resample('W')['qty']
      .sum()
      .reset_index()
      .rename(columns={'qty': 'qty_total'})
    )

    if len(weekly) < WINDOW_SIZE:
        return None

    scaled = scaler.transform(weekly[['qty_total']])
    last_window = scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
    pred_scaled = model.predict(last_window, verbose=0)
    pred_actual = max(0, scaler.inverse_transform(pred_scaled)[0][0])

    stock_row = stock_per_seller_cat[
        (stock_per_seller_cat['id_seller']       == seller_id) &
        (stock_per_seller_cat['kategori_produk'] == category)
    ]

    stok_aktual  = float(stock_row['stok_aktual'].values[0])  if len(stock_row) > 0 else 0
    minimum_stok = float(stock_row['minimum_stok'].values[0]) if len(stock_row) > 0 else 10

    status = classify_restock(pred_actual, stok_aktual, minimum_stok)

    return {
        'seller_id'       : seller_id,
        'kategori'        : category,
        'prediksi_demand' : round(pred_actual, 2),
        'stok_aktual'     : stok_aktual,
        'minimum_stok'    : minimum_stok,
        'coverage_weeks'  : round(stok_aktual / pred_actual, 2) if pred_actual > 0 else float('inf'),
        'status_restock'  : status,
    }

## 12. Jalankan Prediksi Semua Seller × Kategori

In [ ]:
rekomendasi_list = []

for _, row in seller_cat_pairs.iterrows():
    result = predict_and_recommend(row['id_seller'], row['kategori_produk'])
    if result:
        rekomendasi_list.append(result)

if rekomendasi_list:
    rekomendasi_df = (
        pd.DataFrame(rekomendasi_list)
        .sort_values('coverage_weeks')
        .reset_index(drop=True)
    )
    print(rekomendasi_df.to_string(index=False))

seller_id    kategori  prediksi_demand  stok_aktual  minimum_stok  coverage_weeks status_restock
     U005     Laundry       205.639999     213384.0        3507.0     1037.670044      Stok Aman
     U002     Sembako        81.750000     124426.0        3255.0     1522.010010      Stok Aman
     U002       Snack        57.290001     100102.0        1588.0     1747.430054      Stok Aman
     U003 Frozen Food        45.730000      94934.0        1063.0     2076.149902      Stok Aman
     U004      Bakery        89.339996     192820.0         721.0     2158.189941      Stok Aman
     U003     Makanan        43.119999      99330.0         857.0     2303.739990      Stok Aman
     U006     Fashion        75.820000     199314.0         513.0     2628.750000      Stok Aman
     U001     Minuman        71.580002     195315.0         758.0     2728.639893      Stok Aman


In [ ]:
if rekomendasi_list:
    perlu_perhatian = rekomendasi_df[
        rekomendasi_df['status_restock'].isin(['Restock Segera', 'Monitor'])
    ]
    if not perlu_perhatian.empty:
        print(f"Kategori yang butuh perhatian ({len(perlu_perhatian)} kombinasi):\n")
        print(perlu_perhatian.to_string(index=False))
    else:
        print("Semua seller & kategori dalam kondisi stok aman.")

Semua seller & kategori dalam kondisi stok aman.


In [ ]:
if rekomendasi_list:
    per_seller = (
        rekomendasi_df
        .groupby('seller_id')['status_restock']
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    print("Ringkasan status restock per seller:")
    print(per_seller.to_string(index=False))

Ringkasan status restock per seller:
seller_id  Stok Aman
     U001          1
     U002          2
     U003          2
     U004          1
     U005          1
     U006          1
